In [1]:
import torch

In [2]:
t = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

print(t.shape, t.dtype)
print(t.ndim, t.numel())
print(torch.zeros(2, 3).shape)
print(torch.arange(6).reshape(2, 3))

torch.Size([2, 3]) torch.float32
2 6
torch.Size([2, 3])
tensor([[0, 1, 2],
        [3, 4, 5]])


In [3]:
print(t.sum(dim=0).tolist())
print(t.sum(dim=1).tolist())

[5.0, 7.0, 9.0]
[6.0, 15.0]


In [4]:
print(t.mean().item())
print((t**2).tolist())
print((t @ t.T).tolist())

3.5
[[1.0, 4.0, 9.0], [16.0, 25.0, 36.0]]
[[14.0, 32.0], [32.0, 77.0]]


In [5]:
print(t[0, 1].item())
print(t[:, 1].tolist())
print(t[t > 3].tolist())

2.0
[2.0, 5.0]
[4.0, 5.0, 6.0]


In [6]:
print(t.reshape(3, 2).tolist())
print(t.reshape(-1).tolist())
print(t.reshape(3, -1).tolist())

[[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]
[1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
[[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]


In [10]:
import numpy as np

In [8]:
print(torch.tensor([1, 2, 3]).dtype)
print((torch.tensor([1, 2, 3]) / 2).dtype)

torch.int64
torch.float32


In [ ]:
print(torch.tensor([1, 2, 3]).float().mean().item())

In [11]:
a = np.array([1.0, 2.0, 3.0])
t1 = torch.from_numpy(a)
t2 = torch.tensor(a)

a[0] = 100

In [ ]:
print(t1.tolist(), t2.tolist())
print(t1.dtype)

[100.0, 2.0, 3.0] [1.0, 2.0, 3.0]
torch.float64


In [17]:
print(torch.ones(3, 1).tolist())
print(torch.arange(4))

[[1.0], [1.0], [1.0]]
tensor([0, 1, 2, 3])


In [ ]:
print(torch.ones(3, 1) + torch.arange(4))

tensor([[1., 2., 3., 4.],
        [1., 2., 3., 4.],
        [1., 2., 3., 4.]])


In [20]:
print(t.reshape(2, -1).shape)

torch.Size([2, 3])


In [ ]:
x = torch.tensor(2.0)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

y = w * x + b
loss = (y - 10) ** 2

print(y.item(), loss.item())
loss.backward()
print(w.grad.item(), b.grad.item())
print(x.grad)

7.0 9.0
-12.0 -6.0
None


In [24]:
loss = (w * x + b - 10) ** 2
loss.backward()
print(w.grad.item(), b.grad.item())

w.grad.zero_()
b.grad.zero_()
print(w.grad.item(), b.grad.item())

-24.0 -12.0
0.0 0.0


In [ ]:
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

for step in range(3):
    loss = (w * x + b - 10) ** 2
    loss.backward()
    with torch.no_grad():
        w -= 0.1 * w.grad
        b -= 0.1 * b.grad
    w.grad.zero_()
    b.grad.zero_()
    print(step, round(loss.item(), 4), round(w.item(), 4), round(b.item(), 4))

0 9.0 4.2 1.6
1 0.0 4.2 1.6
2 0.0 4.2 1.6


In [27]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x
y.backward()
print(x.grad.item())

7.0


In [32]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
t = torch.tensor(4.0, requires_grad=True)
x = torch.tensor(3.0, requires_grad=True)

for step in range(2):
    loss = (w * x + b - t) ** 2
    loss.backward()
    with torch.no_grad():
        w -= 0.05 * w.grad
        b -= 0.05 * b.grad
    w.grad.zero_()
    b.grad.zero_()
    print(step, round(loss.item(), 4), round(w.item(), 4), round(b.item(), 4))

0 4.0 1.4 -0.2
1 0.0 1.4 -0.2


In [33]:
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
lin = nn.Linear(13, 16)
print(tuple(lin.weight.shape), tuple(lin.bias.shape))
xb = torch.randn(5, 13)
print(tuple(lin(xb).shape))
print(torch.allclose(lin(xb), xb @ lin.weight.T + lin.bias))
print(sum(p.numel() for p in lin.parameters()))

(16, 13) (16,)
(5, 16)
True
224


In [36]:
class WineNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(13, 16)
        self.fc2 = nn.Linear(16, 3)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


torch.manual_seed(0)
model = WineNet()

xb = torch.randn(5, 13)

print(tuple(model(xb).shape))
print(sum(p.numel() for p in model.parameters()))
for name, p in model.named_parameters():
    print(name, tuple(p.shape))

(5, 3)
275
fc1.weight (16, 13)
fc1.bias (16,)
fc2.weight (3, 16)
fc2.bias (3,)


In [41]:
torch.manual_seed(0)
sec = nn.Sequential(nn.Linear(13, 16), nn.ReLU(), nn.Linear(16, 3))
print(torch.allclose(model(xb), sec(xb)))
print(list(model.state_dict().keys()))

True
['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias']


In [ ]:
torch.save(model.state_dict(), "wine.pt")
torch.manual_seed(0)
new = WineNet()
print(torch.allclose(model(xb), new(xb)))
new.load_state_dict(torch.load("wine.pt", weights_only=True))
print(torch.allclose(model(xb), new(xb)))

True
True


In [43]:
torch.manual_seed(0)
lin = nn.Linear(4, 2)
print(tuple(lin.weight.shape))
print(sum(p.numel() for p in lin.parameters()))

(2, 4)
10


In [44]:
torch.manual_seed(0)
sec = nn.Sequential(
    nn.Linear(13, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 3)
)
print(sum(p.numel() for p in sec.parameters()))

1027


In [47]:
class WineNet2(nn.Module):
    def __init__(self, hid=8):
        super().__init__()
        self.fc1 = nn.Linear(13, hid)
        self.fc2 = nn.Linear(hid, 3)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


model = WineNet2(8)

print(sum(p.numel() for p in model.parameters()))

139


In [49]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
sc = StandardScaler().fit(X_train)

A = torch.tensor(sc.transform(X_train), dtype=torch.float32)
B = torch.tensor(sc.transform(X_test), dtype=torch.float32)
ya, yb = torch.tensor(y_train), torch.tensor(y_test)

ds = TensorDataset(A, ya)
dl = DataLoader(ds, batch_size=32, shuffle=False)
print(A.shape, A.dtype, ya.dtype)
print([len(xb) for xb, _ in dl])

torch.Size([124, 13]) torch.float32 torch.int64
[32, 32, 32, 28]


In [50]:
def train(lr=0.1, epochs=50, bs=32, seed=0, zero=True):
    torch.manual_seed(seed)
    m = WineNet()
    opt = torch.optim.SGD(m.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    dl = DataLoader(ds, batch_size=bs, shuffle=True, generator=g)
    hist = []
    for ep in range(epochs):
        m.train()
        total = 0
        for xb, yb_ in dl:
            loss = F.cross_entropy(m(xb), yb_)
            if zero:
                opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(xb)
        hist.append(total / len(ds))
    return m, hist

In [53]:
def accuracy(m, X, y):
    m.eval()
    with torch.no_grad():
        return (m(X).argmax(1) == y).float().mean().item()


model, hist = train()
print([round(h, 3) for h in hist[::5]])

tr = accuracy(model, A, ya)
te = accuracy(model, B, yb)
print(round(tr, 4))
print(round(te, 4))

[0.936, 0.378, 0.17, 0.101, 0.07, 0.054, 0.043, 0.035, 0.03, 0.025]
1.0
0.9815


In [56]:
for zero in (True, False):
    accs, ls = [], []
    for s in range(5):
        m, h = train(seed=s, zero=zero)
        accs.append(accuracy(m, B, yb))
        ls.append(h[-1])

    print(zero, round(sum(ls) / 5, 4))
    print(round(sum(accs) / 5, 4))
    print(round(min(accs) / 5, 4))

True 0.0227
0.9926
0.1963
False 0.0
0.963
0.1889


In [57]:
def train(lr=0.01, epochs=100, bs=32, seed=5, zero=True):
    torch.manual_seed(seed)
    m = WineNet()
    opt = torch.optim.SGD(m.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    dl = DataLoader(ds, batch_size=bs, shuffle=True, generator=g)
    hist = []
    for ep in range(epochs):
        m.train()
        total = 0
        for xb, yb_ in dl:
            loss = F.cross_entropy(m(xb), yb_)
            if zero:
                opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(xb)
        hist.append(total / len(ds))
    return m, hist

In [58]:
def accuracy(m, X, y):
    m.eval()
    with torch.no_grad():
        return (m(X).argmax(1) == y).float().mean().item()


model, hist = train()
print([round(h, 3) for h in hist[::5]])

tr = accuracy(model, A, ya)
te = accuracy(model, B, yb)
print(round(tr, 4))
print(round(te, 4))

[1.146, 1.042, 0.95, 0.865, 0.784, 0.707, 0.634, 0.565, 0.503, 0.448, 0.399, 0.356, 0.319, 0.287, 0.26, 0.237, 0.216, 0.199, 0.184, 0.171]
0.9839
1.0


In [59]:
def train(lr=0.01, epochs=100, bs=124, seed=5, zero=True):
    torch.manual_seed(seed)
    m = WineNet()
    opt = torch.optim.SGD(m.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    dl = DataLoader(ds, batch_size=bs, shuffle=True, generator=g)
    hist = []
    for ep in range(epochs):
        m.train()
        total = 0
        for xb, yb_ in dl:
            loss = F.cross_entropy(m(xb), yb_)
            if zero:
                opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(xb)
        hist.append(total / len(ds))
    return m, hist

In [ ]:
def accuracy(m, X, y):
    m.eval()
    with torch.no_grad():
        return (m(X).argmax(1) == y).float().mean().item()


model, hist = train()
print([round(h, 3) for h in hist[::5]])

tr = accuracy(model, A, ya)
te = accuracy(model, B, yb)
print(round(tr, 4))
print(round(te, 4))

[1.153, 1.125, 1.099, 1.073, 1.049, 1.025, 1.001, 0.978, 0.956, 0.934, 0.913, 0.891, 0.87, 0.85, 0.829, 0.809, 0.789, 0.769, 0.75, 0.731]
0.9194
0.963
